In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor, Pool
warnings.filterwarnings('ignore')
sns.set_theme(style="white", font_scale=1.2)

def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'database.xlsx').exists():
            return candidate
    raise FileNotFoundError('Could not locate database.xlsx in the current directory or its parents.')

ROOT = find_project_root()
OUT = ROOT / 'outputs' / 'grid_search'
OUT.mkdir(parents=True, exist_ok=True)

SPLIT, CV, SEED =10, 5, 42

F = ['AM', 'AMc', 'Pr', 'Prc', 'Sp', 'Sp2', 'MSP', 'PM', 'CT', 'Ct', 'RT', 'Rt', 'RH', 'T', 'P', 'GHSV', 'Inert', 'H2/CO2']
C = ['AM', 'Pr', 'Sp', 'Sp2', 'PM']
N = [x for x in F if x not in C]



In [ ]:
data = pd.read_excel(ROOT / 'database.xlsx')
dx1 = data[F].copy()
dy1 = pd.to_numeric(data['Rate'], errors='coerce')

dx1[N] = SimpleImputer(strategy='mean').fit_transform(dx1[N])
dx1[C] = SimpleImputer(strategy='most_frequent').fit_transform(dx1[C])

df_encoded = pd.get_dummies(dx1, columns=C, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(df_encoded, dy1, test_size=0.2, random_state=SPLIT)

print(df_encoded)


In [ ]:
xgb_regressor = XGBRegressor(
    random_state=SEED,
    learning_rate=0.1,
    n_estimators=100,
    max_depth=3,
    subsample=0.6,
    objective='reg:squarederror'
)

param_grid_xgb = {
    'n_estimators': list(range(500, 1501, 100)),
    'max_depth': list(range(6, 25, 3)),
    'learning_rate': [i / 100 for i in range(5, 26, 5)]
}

grid_search_xgb = GridSearchCV(
    estimator=xgb_regressor,
    param_grid=param_grid_xgb,
    scoring='neg_mean_squared_error',
    cv=CV,
    n_jobs=-1,
    verbose=1
)

grid_search_xgb.fit(X_train, y_train)

print("Best parameters found: ", grid_search_xgb.best_params_)
print("Best negative mean squared error score: ", grid_search_xgb.best_score_)

XGB = grid_search_xgb.best_estimator_
y_pred_XGB = XGB.predict(X_test)

y_train_pred_XGB = XGB.predict(X_train)
trainrmse = mean_squared_error(y_train, y_train_pred_XGB, squared=False)
trainr2 = r2_score(y_train, y_train_pred_XGB)
std_dev_obs = np.std(y_test)
std_dev_pred = np.std(y_pred_XGB)
correlation = np.corrcoef(y_test, y_pred_XGB)[0, 1]
rmse = mean_squared_error(y_test, y_pred_XGB, squared=False)
r2 = r2_score(y_test, y_pred_XGB)

In [ ]:
metrics_df = pd.DataFrame({
    'Model': ['XGBoost'],
    'Train RMSE':[trainrmse],
    'Train r2':[trainr2],    
    'Standard Deviation (Pred)': [std_dev_pred],
    'Standard Deviation (Observed)': [std_dev_obs],
    'Correlation': [correlation],
    'RMSE': [rmse],
    'R2 Score': [r2]
})

metrics_df

In [ ]:
rf_regressor = RandomForestRegressor(random_state=SEED)
param_grid_rf = {
    'n_estimators': list(range(500, 1501, 100)),
    'max_depth': list(range(6, 25, 3)),
    'min_samples_split': list(range(2, 6, 1))
}
grid_search_rf = GridSearchCV(
    estimator=rf_regressor,
    param_grid=param_grid_rf,
    scoring='neg_mean_squared_error',
    cv=CV,
    n_jobs=-1,
    verbose=1
)
grid_search_rf.fit(X_train, y_train)

print("RF Best parameters: ", grid_search_rf.best_params_)
print("RF Best score: ", grid_search_rf.best_score_)

RF = grid_search_rf.best_estimator_
y_pred_RF = RF.predict(X_test)
y_train_pred_RF = RF.predict(X_train)

new_row_rf = pd.DataFrame({
    'Model': ['RF'],
    'Train RMSE': [mean_squared_error(y_train, y_train_pred_RF, squared=False)],
    'Train r2': [r2_score(y_train, y_train_pred_RF)],
    'Standard Deviation (Pred)': [np.std(y_pred_RF)],
    'Standard Deviation (Observed)': [std_dev_obs],
    'Correlation': [np.corrcoef(y_test, y_pred_RF)[0, 1]],
    'RMSE': [mean_squared_error(y_test, y_pred_RF, squared=False)],
    'R2 Score': [r2_score(y_test, y_pred_RF)]
})
metrics_df = pd.concat([metrics_df, new_row_rf], ignore_index=True)
metrics_df

In [ ]:

metrics_df

In [ ]:
et_regressor = ExtraTreesRegressor(random_state=SEED)
param_grid_et = {
    'n_estimators': list(range(500, 1501, 100)),
    'max_depth': list(range(6, 25, 3)),
    'min_samples_split': list(range(2, 6, 1))
}
grid_search_et = GridSearchCV(
    estimator=et_regressor,
    param_grid=param_grid_et,
    scoring='neg_mean_squared_error',
    cv=CV,
    n_jobs=-1,
    verbose=1
)
grid_search_et.fit(X_train, y_train)

print("ET Best parameters: ", grid_search_et.best_params_)
print("ET Best score: ", grid_search_et.best_score_)

ET = grid_search_et.best_estimator_
y_pred_ET = ET.predict(X_test)
y_train_pred_ET = ET.predict(X_train)

new_row_et = pd.DataFrame({
    'Model': ['ET'],
    'Train RMSE': [mean_squared_error(y_train, y_train_pred_ET, squared=False)],
    'Train r2': [r2_score(y_train, y_train_pred_ET)],
    'Standard Deviation (Pred)': [np.std(y_pred_ET)],
    'Standard Deviation (Observed)': [std_dev_obs],
    'Correlation': [np.corrcoef(y_test, y_pred_ET)[0, 1]],
    'RMSE': [mean_squared_error(y_test, y_pred_ET, squared=False)],
    'R2 Score': [r2_score(y_test, y_pred_ET)]
})
metrics_df = pd.concat([metrics_df, new_row_et], ignore_index=True)

In [ ]:
lgbm_regressor = LGBMRegressor(random_state=SEED, verbosity=-1)
param_grid_lgbm = {
    'n_estimators': list(range(500, 1501, 100)),
    'max_depth': list(range(6, 25, 3)),
    'learning_rate': [i / 100 for i in range(5, 26, 5)]
}
grid_search_lgbm = GridSearchCV(
    estimator=lgbm_regressor,
    param_grid=param_grid_lgbm,
    scoring='neg_mean_squared_error',
    cv=CV,
    n_jobs=-1,
    verbose=1
)
grid_search_lgbm.fit(X_train, y_train)

print("LGBM Best parameters: ", grid_search_lgbm.best_params_)
print("LGBM Best score: ", grid_search_lgbm.best_score_)

LGBM = grid_search_lgbm.best_estimator_
y_pred_LGBM = LGBM.predict(X_test)
y_train_pred_LGBM = LGBM.predict(X_train)

new_row_lgbm = pd.DataFrame({
    'Model': ['LGBM'],
    'Train RMSE': [mean_squared_error(y_train, y_train_pred_LGBM, squared=False)],
    'Train r2': [r2_score(y_train, y_train_pred_LGBM)],
    'Standard Deviation (Pred)': [np.std(y_pred_LGBM)],
    'Standard Deviation (Observed)': [std_dev_obs],
    'Correlation': [np.corrcoef(y_test, y_pred_LGBM)[0, 1]],
    'RMSE': [mean_squared_error(y_test, y_pred_LGBM, squared=False)],
    'R2 Score': [r2_score(y_test, y_pred_LGBM)]
})
metrics_df = pd.concat([metrics_df, new_row_lgbm], ignore_index=True)


In [ ]:
cb_ohe_regressor = CatBoostRegressor(random_seed=SEED, verbose=False, allow_writing_files=False)
param_grid_cb = {
    'learning_rate': [i / 100 for i in range(5, 41, 5)],
    'depth': list(range(6, 16, 1))
}
grid_search_cb_ohe = GridSearchCV(
    estimator=cb_ohe_regressor,
    param_grid=param_grid_cb,
    scoring='neg_mean_squared_error',
    cv=CV,
    n_jobs=-1,
    verbose=1
)
grid_search_cb_ohe.fit(X_train, y_train)

print("CatBoost-OHE Best parameters: ", grid_search_cb_ohe.best_params_)
print("CatBoost-OHE Best score: ", grid_search_cb_ohe.best_score_)

CB_OHE = grid_search_cb_ohe.best_estimator_
y_pred_CB_OHE = CB_OHE.predict(X_test)
y_train_pred_CB_OHE = CB_OHE.predict(X_train)

new_row_cb_ohe = pd.DataFrame({
    'Model': ['CatBoost-OHE'],
    'Train RMSE': [mean_squared_error(y_train, y_train_pred_CB_OHE, squared=False)],
    'Train r2': [r2_score(y_train, y_train_pred_CB_OHE)],
    'Standard Deviation (Pred)': [np.std(y_pred_CB_OHE)],
    'Standard Deviation (Observed)': [std_dev_obs],
    'Correlation': [np.corrcoef(y_test, y_pred_CB_OHE)[0, 1]],
    'RMSE': [mean_squared_error(y_test, y_pred_CB_OHE, squared=False)],
    'R2 Score': [r2_score(y_test, y_pred_CB_OHE)]
})
metrics_df = pd.concat([metrics_df, new_row_cb_ohe], ignore_index=True)
metrics_df